In [ ]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from datetime import date
import marineHeatWaves as mhw
import netCDF4 as nc
import datetime
import matplotlib.pyplot as plt 
from tqdm import notebook
from scipy.interpolate import griddata
#import taichi as ti
#ti.init(arch=ti.cpu)
#import numba
#from numba import jit

from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor

In [ ]:
file=r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_6.nc"
daset=nc.Dataset(file)
#print(list(daset.variables.keys())[:])

In [ ]:
lat_12=np.array(daset['latitude'])
lon_12=np.array(daset['longitude'])
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{lat_12[1]-lat_12[0]}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')

In [ ]:
lat_4=np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/lats.npy')
lon_4=np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/lons.npy')
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')

In [ ]:
thetao1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_6.nc")['thetao'])
thetao2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_7.nc")['thetao'])
thetao3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_8.nc")['thetao'])

In [ ]:
thetaos=np.concatenate([thetao1,thetao2,thetao3],axis=0)

In [ ]:
lat_4_ind=(lat_4>=35)&(lat_4<=50)
lon_4_ind=(lon_4>=165)&(lon_4<=200)
Lon_4,Lat_4=np.meshgrid(lon_4[lon_4_ind],lat_4[lat_4_ind])
Lon_12,Lat_12=np.meshgrid(lon_12,lat_12)
points=np.array([Lon_12.flatten(),Lat_12.flatten()]).T

def grid(dat):
    global points,Lon_4,Lat_4
    return griddata(points,dat.flatten(),(Lon_4,Lat_4),'cubic')

def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=15)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans

In [ ]:
thetaos.shape

In [ ]:
%%time
thetaos_new=np.array(list(map(list_map,thetaos[0:2,:,:,:])))

In [30]:
%%time
thetao1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_6.nc")['thetao'])
thetao2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_7.nc")['thetao'])
thetao3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2021_8.nc")['thetao'])
thetaos=np.concatenate([thetao1,thetao2,thetao3],axis=0)
pool = ProcessPoolExecutor(max_workers=4)
thetaos_new=np.array(list(pool.map(list_map,thetaos[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/thetaos_2021.npy',thetaos_new)

CPU times: user 7.91 s, sys: 6.76 s, total: 14.7 s
Wall time: 7min 7s


In [28]:
%%time
thetao1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2022_6.nc")['thetao'])
thetao2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2022_7.nc")['thetao'])
thetao3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/thetao_2022_8.nc")['thetao'])
thetaos=np.concatenate([thetao1,thetao2,thetao3],axis=0)

pool = ProcessPoolExecutor(max_workers=4)
thetaos_new=np.array(list(pool.map(list_map,thetaos[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/thetaos_2022.npy',thetaos_new)



CPU times: user 8 s, sys: 6.8 s, total: 14.8 s
Wall time: 7min 2s


In [29]:
%%time
uo1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2021_6.nc")['uo'])
uo2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2021_7.nc")['uo'])
uo3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2021_8.nc")['uo'])
uos=np.concatenate([uo1,uo2,uo3],axis=0)
pool = ProcessPoolExecutor(max_workers=4)
uos_new=np.array(list(pool.map(list_map,uos[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/uos_2021.npy',uos_new)

CPU times: user 7.92 s, sys: 6.87 s, total: 14.8 s
Wall time: 7min 1s


In [32]:
%%time
uo1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2022_6.nc")['uo'])
uo2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2022_7.nc")['uo'])
uo3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/uo_2022_8.nc")['uo'])
uos=np.concatenate([uo1,uo2,uo3],axis=0)

pool = ProcessPoolExecutor(max_workers=4)
uos_new=np.array(list(pool.map(list_map,uos[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/uos_2022.npy',uos_new)


vo1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2021_6.nc")['vo'])
vo2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2021_7.nc")['vo'])
vo3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2021_8.nc")['vo'])
vo=np.concatenate([vo1,vo2,vo3],axis=0)

pool = ProcessPoolExecutor(max_workers=4)
vo_new=np.array(list(pool.map(list_map,vo[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/vo_2021.npy',vo_new)

vo1=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2022_6.nc")['vo'])
vo2=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2022_7.nc")['vo'])
vo3=np.array(nc.Dataset(r"/lustre/home/yuhanxue/data/Glorys_new/NEW_GLORYS24/vo_2022_8.nc")['vo'])
vo=np.concatenate([vo1,vo2,vo3],axis=0)

pool = ProcessPoolExecutor(max_workers=4)
vo_new=np.array(list(pool.map(list_map,vo[:,:,:,:])))
del pool
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/vo_2022.npy',vo_new)

CPU times: user 23.8 s, sys: 21.7 s, total: 45.6 s
Wall time: 21min 7s
